# Proposta Wesley

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcelo7bastos/relatorio_dados_damei/blob/main/notebooks/proposta_wesley.ipynb)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

In [ ]:
#### Atenção - Precisa indicar a pasta correta!!!!
# Diretórios principais
RAW_DIR = '/content/drive/MyDrive/DAMEI_Relatorio_Dados/dados_brutos/dado_atual'
OUTPUT_DIR = '/content/drive/MyDrive/DAMEI_Relatorio_Dados/dados_brutos/dados_atual'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Listar todos os arquivos .xlsx no RAW_DIR
excel_files = glob.glob(os.path.join(RAW_DIR, '*.xlsx'))
print(f'Foram encontrados {len(excel_files)} arquivos:')
for f in excel_files:
    print('  •', os.path.basename(f))

In [ ]:
# Listar abas de todos os arquivos Excel
for f in excel_files:
    print(f"\nArquivo: {os.path.basename(f)}")
    try:
        xls = pd.ExcelFile(f)
        print("Abas encontradas:", xls.sheet_names)
    except Exception as e:
        print("Erro ao abrir:", e)

In [ ]:
import os

# Dicionário para armazenar os arquivos encontrados
arquivos_encontrados = {
    "file_name_caf": None,
    "file_name_ater": None,
    "file_name_pronaf": None,
    "file_name_mais_alimentos": None,
    "file_name_pncf": None,
    "file_name_garantia_safra": None,
    "file_name_selo_bio": None,
}

# Loop para identificar cada arquivo por parte do nome
for f in excel_files:
    nome_base = os.path.basename(f).lower()
    if "caf" in nome_base:
        arquivos_encontrados["file_name_caf"] = f
    elif "ater" in nome_base:
        arquivos_encontrados["file_name_ater"] = f
    elif "pronaf" in nome_base:
        arquivos_encontrados["file_name_pronaf"] = f
    elif "mais_alimentos" in nome_base:
        arquivos_encontrados["file_name_mais_alimentos"] = f
    elif "pncf" in nome_base:
        arquivos_encontrados["file_name_pncf"] = f
    elif "garantia-safra" in nome_base:
        arquivos_encontrados["file_name_garantia_safra"] = f
    elif "selo_bio" in nome_base:
        arquivos_encontrados["file_name_selo_bio"] = f

# Exibir resultados
for var_name, arquivo in arquivos_encontrados.items():
    if arquivo:
        print(f"{var_name} = '{arquivo}'")
    else:
        print(f"{var_name} não encontrado.")

#CAF


In [ ]:
#Carga Dados CAF
df_caf_dap = pd.read_excel(
    arquivos_encontrados['file_name_caf'],
    sheet_name='TOTALIZADORES',  # Corrigido para maiúsculas conforme inspecionado anteriormente
    engine='openpyxl',
    na_values=['NA', 'na', ''],
    skiprows=6,
    nrows=27
)
df_caf_dap

df_caf_dap.columns.tolist()

In [ ]:
#Tratamento dos dados
# 1. Remover espaços extras dos NOMES das colunas
df_caf_dap.columns = df_caf_dap.columns.str.strip()

# 2. Definir colunas como NÚMERO
colunas_numericas = [
    'Soma de CAFs PF ATIVO',
    'Soma de CAFs PJ ATIVO',
    'Soma de QUANTIDADE DE MULHERES EM CAF ATIVO',
    'Soma de QUANTIDADE DE HOMENS EM CAF ATIVO'
]

# Loop para converter cada uma delas para número
for col in colunas_numericas:
    df_caf_dap[col] = pd.to_numeric(df_caf_dap[col], errors='coerce')

# Verificando se a conversão deu certo
print(df_caf_dap[colunas_numericas].dtypes)

# 3. Definir colunas como texto
colunas_texto = [
    'dt_referencia',
    'dt_geracao',
    'cod_ibge_estados',
    'uf'
]

# Loop para converter cada uma delas para texto (string)
for col in colunas_texto:
    df_caf_dap[col] = df_caf_dap[col].astype(str)

# Verificando se a conversão deu certo
print(df_caf_dap[colunas_texto].dtypes)

# PRONAF

In [ ]:
#Carga Dados PRONAF
df_pronaf = pd.read_excel(
    '/content/drive/MyDrive/DAMEI_Relatorio_Dados/dados_brutos/dado_atual/pronaf_gaia_20260414.xlsx',
    sheet_name='Totalizadores',
    engine='openpyxl',
    na_values=['NA', 'na', ''],
    #skiprows=6,
    #nrows=27
)
df_pronaf

df_pronaf.columns.tolist()

In [ ]:
#Tratamento dos dados
# 1. Remover espaços extras dos NOMES das colunas
df_pronaf.columns = df_pronaf.columns.str.strip()

# 2. Definir colunas como texto
colunas_texto = [
    'dt_referencia',
    'dt_geracao',
    'uf',
    'cod_ibge_uf',
    'ANO'
]
# Loop para converter cada uma delas para texto (string)
for col in colunas_texto:
    df_pronaf[col] = df_pronaf[col].astype(str)

 # 3. Definir colunas como NÚMERO
colunas_numericas = [
  'qtd_contratos_1_Feminino',
 'qtd_contratos_1_Masculino',
 'qtd_contratos_1_Sem_Identificacao',
 'valor_total_contratos_1_Feminino',
 'valor_total_contratos_1_Masculino',
 'valor_total_contratos_1_Sem_Identificacao',
 'qtd_operacoes_1_Feminino',
 'qtd_operacoes_1_Masculino',
 'qtd_operacoes_1_Sem_Identificacao',
 'ticket_medio_1_Feminino',
 'ticket_medio_1_Masculino',
 'ticket_medio_1_Sem_Identificacao']

 # Loop para converter cada uma delas para número
for col in colunas_numericas:
    df_pronaf[col] = pd.to_numeric(df_pronaf[col], errors='coerce')

# Verificando se a conversão deu certo
print(df_pronaf[colunas_numericas].dtypes)

# Proposta Texto


In [ ]:
!pip install python-docx

In [ ]:
import pandas as pd
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from google.colab import files

In [ ]:
# Filtrar ambas as bases para Minas Gerais (MG)
df_caf_mg = df_caf_dap[df_caf_dap['uf'].str.upper() == 'MG'].copy()
df_pronaf_mg = df_pronaf[df_pronaf['uf'].str.upper() == 'MG'].copy()

In [ ]:
# Criar o documento
doc = Document()

# Título
titulo = doc.add_heading('Relatório de Dados Teste: Minas Gerais (MG)', level=0)
titulo.alignment = WD_ALIGN_PARAGRAPH.CENTER

# --- SEÇÃO 1: CAF (Inscrições e Gênero) ---
doc.add_heading('1. Situação do CAF (Cadastro Nacional da Agricultura Familiar)', level=1)

# Cálculos CAF MG
total_caf_pf = df_caf_mg['Soma de CAFs PF ATIVO'].sum()
total_caf_pj = df_caf_mg['Soma de CAFs PJ ATIVO'].sum()
mulheres_caf = df_caf_mg['Soma de QUANTIDADE DE MULHERES EM CAF ATIVO'].sum()
homens_caf = df_caf_mg['Soma de QUANTIDADE DE HOMENS EM CAF ATIVO'].sum()

p = doc.add_paragraph()
p.add_run('Em Minas Gerais, o panorama de CAF ativos apresenta os seguintes números:').italic = True
doc.add_paragraph(f'• Total de CAFs Pessoa Física: {total_caf_pf:,.0f}'.replace(',', '.'))
doc.add_paragraph(f'• Total de CAFs Pessoa Jurídica: {total_caf_pj:,.0f}'.replace(',', '.'))
doc.add_paragraph(f'• Participação Feminina (CAF): {mulheres_caf:,.0f} mulheres'.replace(',', '.'))
doc.add_paragraph(f'• Participação Masculina (CAF): {homens_caf:,.0f} homens'.replace(',', '.'))

# --- SEÇÃO 2: PRONAF (Contratos e Valores) ---
doc.add_heading('2. Desempenho do PRONAF', level=1)

# Cálculos Pronaf MG
contratos_fem = df_pronaf_mg['qtd_contratos_1_Feminino'].sum()
contratos_masc = df_pronaf_mg['qtd_contratos_1_Masculino'].sum()
valor_fem = df_pronaf_mg['valor_total_contratos_1_Feminino'].sum()
valor_masc = df_pronaf_mg['valor_total_contratos_1_Masculino'].sum()

doc.add_paragraph('Quanto ao acesso ao crédito via Pronaf, destacam-se os seguintes indicadores:')

# Tabela de Crédito por Gênero
table = doc.add_table(rows=1, cols=3)
table.style = 'Table Grid'
hdr_cells = table.rows[0].cells
hdr_cells[0].text = 'Público'
hdr_cells[1].text = 'Qtd. Contratos'
hdr_cells[2].text = 'Valor Total (R$)'

# Linha Feminino
row_fem = table.add_row().cells
row_fem[0].text = 'Feminino'
row_fem[1].text = f'{contratos_fem:,.0f}'.replace(',', '.')
row_fem[2].text = f'R$ {valor_fem:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

# Linha Masculino
row_masc = table.add_row().cells
row_masc[0].text = 'Masculino'
row_masc[1].text = f'{contratos_masc:,.0f}'.replace(',', '.')
row_masc[2].text = f'R$ {valor_masc:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')
# --- CÁLCULO DOS TOTAIS ---
total_contratos = contratos_fem + contratos_masc
total_valor = valor_fem + valor_masc

# --- INSERÇÃO DA LINHA DE TOTAL NA TABELA ---
row_total = table.add_row().cells

# 1. Célula "Total" (em negrito)
p = row_total[0].paragraphs[0]
run = p.add_run('Total')
run.bold = True

# 2. Célula de Quantidade Total (em negrito e formatada)
p = row_total[1].paragraphs[0]
run = p.add_run(f'{total_contratos:,.0f}'.replace(',', '.'))
run.bold = True

# 3. Célula de Valor Total (em negrito e formatada como Moeda R$)
p = row_total[2].paragraphs[0]
texto_moeda = f'R$ {total_valor:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')
run = p.add_run(texto_moeda)
run.bold = True

# Salvar e Baixar
nome_relatorio = 'Relatorio_MG_CAF_PRONAF.docx'
doc.save(nome_relatorio)
files.download(nome_relatorio)

print("Relatório de MG gerado e pronto para download!")

In [ ]:
# --- SEÇÃO 1: CAF (Inscrições e Gênero) ---
doc.add_heading('1. Situação do CAF (MG vs Brasil)', level=1)

# Cálculos CAF MG
total_caf_pf = df_caf_mg['Soma de CAFs PF ATIVO'].sum()
total_caf_pj = df_caf_mg['Soma de CAFs PJ ATIVO'].sum()
mulheres_caf = df_caf_mg['Soma de QUANTIDADE DE MULHERES EM CAF ATIVO'].sum()
homens_caf = df_caf_mg['Soma de QUANTIDADE DE HOMENS EM CAF ATIVO'].sum()

# Cálculos CAF BRASIL
br_caf_pf = df_caf_dap['Soma de CAFs PF ATIVO'].sum()
br_caf_pj = df_caf_dap['Soma de CAFs PJ ATIVO'].sum()
br_mulheres_caf = df_caf_dap['Soma de QUANTIDADE DE MULHERES EM CAF ATIVO'].sum()
br_homens_caf = df_caf_dap['Soma de QUANTIDADE DE HOMENS EM CAF ATIVO'].sum()

# Função auxiliar para evitar erro de divisão por zero
def calc_pct(mg, br):
    return (mg / br * 100) if br > 0 else 0

p = doc.add_paragraph()
p.add_run('Em Minas Gerais, o panorama de CAF ativos em comparação ao Brasil apresenta os seguintes números:').italic = True

# Tabela CAF
table_caf = doc.add_table(rows=1, cols=4)
table_caf.style = 'Table Grid'
hdr_caf = table_caf.rows[0].cells
hdr_caf[0].text = 'Indicador'
hdr_caf[1].text = 'Minas Gerais'
hdr_caf[2].text = 'Brasil'
hdr_caf[3].text = '% Part. (MG/BR)'

# Função auxiliar para adicionar linhas na tabela CAF
def add_row_caf(tabela, nome, mg, br):
    row = tabela.add_row().cells
    row[0].text = nome
    row[1].text = f'{mg:,.0f}'.replace(',', '.')
    row[2].text = f'{br:,.0f}'.replace(',', '.')
    row[3].text = f'{calc_pct(mg, br):.1f}%'.replace('.', ',')

add_row_caf(table_caf, 'CAFs Pessoa Física', total_caf_pf, br_caf_pf)
add_row_caf(table_caf, 'CAFs Pessoa Jurídica', total_caf_pj, br_caf_pj)
add_row_caf(table_caf, 'Participação Feminina', mulheres_caf, br_mulheres_caf)
add_row_caf(table_caf, 'Participação Masculina', homens_caf, br_homens_caf)


# --- SEÇÃO 2: PRONAF (Contratos e Valores) ---
doc.add_heading('2. Desempenho do PRONAF (MG vs Brasil)', level=1)

# Cálculos Pronaf MG
contratos_fem = df_pronaf_mg['qtd_contratos_1_Feminino'].sum()
contratos_masc = df_pronaf_mg['qtd_contratos_1_Masculino'].sum()
valor_fem = df_pronaf_mg['valor_total_contratos_1_Feminino'].sum()
valor_masc = df_pronaf_mg['valor_total_contratos_1_Masculino'].sum()

# Cálculos Pronaf BRASIL
br_contratos_fem = df_pronaf['qtd_contratos_1_Feminino'].sum()
br_contratos_masc = df_pronaf['qtd_contratos_1_Masculino'].sum()
br_valor_fem = df_pronaf['valor_total_contratos_1_Feminino'].sum()
br_valor_masc = df_pronaf['valor_total_contratos_1_Masculino'].sum()

doc.add_paragraph('Quanto ao acesso ao crédito via Pronaf, destacam-se os seguintes indicadores comparativos:')

# Tabela de Crédito por Gênero (Expandida)
table = doc.add_table(rows=1, cols=7)
table.style = 'Table Grid'
hdr_cells = table.rows[0].cells
cabecalhos = ['Público', 'Qtd. MG', 'Qtd. BR', '% Qtd', 'Valor MG (R$)', 'Valor BR (R$)', '% Valor']
for i, nome in enumerate(cabecalhos):
    hdr_cells[i].text = nome
    hdr_cells[i].paragraphs[0].runs[0].font.bold = True

# Função auxiliar formatar moeda BRL
def formata_moeda(valor):
    return f'{valor:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

# Função auxiliar para adicionar linhas na tabela do PRONAF
def add_row_pronaf(tabela, publico, qtd_mg, qtd_br, val_mg, val_br, is_total=False):
    row = tabela.add_row().cells

    row[0].text = publico
    row[1].text = f'{qtd_mg:,.0f}'.replace(',', '.')
    row[2].text = f'{qtd_br:,.0f}'.replace(',', '.')
    row[3].text = f'{calc_pct(qtd_mg, qtd_br):.1f}%'.replace('.', ',')
    row[4].text = formata_moeda(val_mg)
    row[5].text = formata_moeda(val_br)
    row[6].text = f'{calc_pct(val_mg, val_br):.1f}%'.replace('.', ',')

    if is_total:
        for cell in row:
            for p in cell.paragraphs:
                for run in p.runs:
                    run.bold = True

# Inserindo dados (Feminino, Masculino)
add_row_pronaf(table, 'Feminino', contratos_fem, br_contratos_fem, valor_fem, br_valor_fem)
add_row_pronaf(table, 'Masculino', contratos_masc, br_contratos_masc, valor_masc, br_valor_masc)

# --- CÁLCULO DOS TOTAIS ---
total_contratos_mg = contratos_fem + contratos_masc
total_contratos_br = br_contratos_fem + br_contratos_masc
total_valor_mg = valor_fem + valor_masc
total_valor_br = br_valor_fem + br_valor_masc

# Inserindo a linha de Total
add_row_pronaf(table, 'Total', total_contratos_mg, total_contratos_br, total_valor_mg, total_valor_br, is_total=True)

# Salvar e Baixar
nome_relatorio = 'Relatorio_MG_CAF_PRONAF.docx'
doc.save(nome_relatorio)
files.download(nome_relatorio)

print("Relatório comparativo de MG gerado e pronto para download!")